In [2]:
   %pip install requests pandas matplotlib

^C
Note: you may need to restart the kernel to use updated packages.


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   --------------- ------------------------ 4.2/10.7 MB 23.3 MB/s eta 0:00:01
   ----------------------------- ---------- 7.9/10.7 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 18.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --------------------- ------------------ 5.0/9.3 MB 25.0 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 24.2 MB/s eta 0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 22.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --------------- ------------------------ 5.0/12.6 MB 25.8 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.6 MB 23.6 


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import time
from datetime import datetime, timedelta
import requests
import pandas as pd

BASE = "https://api.carbonintensity.org.uk/intensity"
FMT = "%Y-%m-%dT%H:%MZ"

rows = []
start = datetime(2025, 1, 1)
while start < datetime(2026, 1, 1):
    end = start + timedelta(days=13)          # API allows max 14 days per request
    r = requests.get(f"{BASE}/{start.strftime(FMT)}/{end.strftime(FMT)}",
                     headers={"Accept": "application/json"})
    r.raise_for_status()
    for d in r.json()["data"]:
        rows.append({"time": d["from"],
                     "forecast": d["intensity"].get("forecast"),
                     "actual": d["intensity"].get("actual")})
    start = end
    time.sleep(0.5)                          # be polite to the server

df = pd.DataFrame(rows).drop_duplicates("time")
df["time"] = pd.to_datetime(df["time"])
df = df[df["time"].dt.year == 2025]
df.to_csv("../data/intensity_2025.csv", index=False)   # ../ = go up from notebooks/ to the repo folder
print(df.shape, df.head())


(17520, 3)                        time  forecast  actual
1 2025-01-01 00:00:00+00:00      49.0      55
2 2025-01-01 00:30:00+00:00      52.0      54
3 2025-01-01 01:00:00+00:00      56.0      53
4 2025-01-01 01:30:00+00:00      53.0      53
5 2025-01-01 02:00:00+00:00      53.0      47
